In [0]:
# ============================================================
# Notebook: 08_gold_aggregations
# Purpose : Build business-ready Gold tables from Silver layer.
#           Gold tables support fraud analytics and reporting.
# ============================================================

from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, max as spark_max,
    min as spark_min, when, lit, round
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SILVER_TXN_TABLE = "data_engineering.silver_layer.finshield_silver"
DIM_ACCOUNTS_TABLE = "data_engineering.silver_layer.dim_accounts"
DIM_CUSTOMERS_TABLE = "data_engineering.silver_layer.dim_customers"

GOLD_DAILY_SUMMARY_TABLE = "data_engineering.gold_layer.daily_txn_summary"
GOLD_CUSTOMER_360_TABLE = "data_engineering.gold_layer.customer_360"
GOLD_FRAUD_RISK_TABLE = "data_engineering.gold_layer.fraud_risk_scores"


# ------------------------------------------------------------
# 2. Read Silver tables
# ------------------------------------------------------------

txn_df = spark.table(SILVER_TXN_TABLE)
accounts_df = spark.table(DIM_ACCOUNTS_TABLE)
customers_df = spark.table(DIM_CUSTOMERS_TABLE).filter(col("is_current") == True)

print("Transaction records:", txn_df.count())
print("Account records:", accounts_df.count())
print("Current customer records:", customers_df.count())

# ------------------------------------------------------------
# 3. Gold Table 1: daily_txn_summary
#    Aggregates transaction volume, amount, and fraud metrics
#    by step and transaction type.
# ------------------------------------------------------------

daily_txn_summary_df = txn_df.groupBy(
    "step",
    "transaction_type"
).agg(
    count("*").alias("txn_count"),
    spark_sum("amount").alias("total_amount"),
    avg("amount").alias("avg_amount"),
    spark_min("amount").alias("min_amount"),
    spark_max("amount").alias("max_amount"),
    spark_sum("is_fraud").alias("fraud_txn_count"),
    spark_sum("is_flagged_fraud").alias("flagged_fraud_count")
).withColumn(
    "fraud_rate_pct",
    round((col("fraud_txn_count") / col("txn_count")) * 100, 4)
)

daily_txn_summary_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_DAILY_SUMMARY_TABLE)

print("Gold table created: daily_txn_summary")

spark.table(GOLD_DAILY_SUMMARY_TABLE).display()

# ------------------------------------------------------------
# 4. Prepare enriched transaction base
#    Join transaction facts with account and current customer
#    dimensions.
# ------------------------------------------------------------

enriched_txn_df = txn_df.alias("t") \
    .join(
        accounts_df.alias("a"),
        col("t.origin_account_id") == col("a.account_id"),
        "left"
    ) \
    .join(
        customers_df.alias("c"),
        col("t.origin_account_id") == col("c.account_id"),
        "left"
    )

print("Enriched transaction records:", enriched_txn_df.count())


# ------------------------------------------------------------
# 5. Gold Table 2: customer_360
#    Creates one analytical profile per customer/account.
# ------------------------------------------------------------

customer_360_df = enriched_txn_df.groupBy(
    col("t.origin_account_id").alias("account_id"),
    col("c.customer_id"),
    col("c.full_name"),
    col("c.country"),
    col("c.risk_tier"),
    col("a.account_type"),
    col("a.status").alias("account_status"),
    col("a.credit_score"),
    col("a.kyc_verified"),
    col("a.is_deleted")
).agg(
    count("*").alias("total_transactions"),
    spark_sum("t.amount").alias("total_transaction_amount"),
    avg("t.amount").alias("avg_transaction_amount"),
    spark_max("t.amount").alias("max_transaction_amount"),
    spark_sum("t.is_fraud").alias("fraud_transactions"),
    spark_sum("t.is_flagged_fraud").alias("flagged_fraud_transactions")
).withColumn(
    "fraud_rate_pct",
    round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
)

customer_360_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_CUSTOMER_360_TABLE)

print("Gold table created: customer_360")

spark.table(GOLD_CUSTOMER_360_TABLE).display()

# ------------------------------------------------------------
# 6. Gold Table 3: fraud_risk_scores
#    Creates transaction-level risk score using transaction,
#    account, and customer signals.
# ------------------------------------------------------------

fraud_risk_scores_df = enriched_txn_df.select(
    col("t.transaction_id"),
    col("t.step"),
    col("t.transaction_type"),
    col("t.amount"),
    col("t.origin_account_id"),
    col("t.destination_account_id"),
    col("t.is_fraud"),
    col("t.is_flagged_fraud"),

    col("a.status").alias("account_status"),
    col("a.credit_score"),
    col("a.kyc_verified"),
    col("a.is_deleted"),

    col("c.risk_tier"),
    col("c.country")
).withColumn(
    "amount_risk_score",
    when(col("amount") >= 1000000, lit(30))
    .when(col("amount") >= 500000, lit(20))
    .when(col("amount") >= 100000, lit(10))
    .otherwise(lit(0))
).withColumn(
    "account_status_risk_score",
    when(col("is_deleted") == True, lit(30))
    .when(col("account_status") == "CLOSED", lit(25))
    .when(col("account_status") == "SUSPENDED", lit(20))
    .otherwise(lit(0))
).withColumn(
    "kyc_risk_score",
    when(col("kyc_verified") == False, lit(15)).otherwise(lit(0))
).withColumn(
    "customer_risk_score",
    when(col("risk_tier") == "HIGH", lit(25))
    .when(col("risk_tier") == "MEDIUM", lit(10))
    .otherwise(lit(0))
).withColumn(
    "model_flag_score",
    when(col("is_flagged_fraud") == 1, lit(20)).otherwise(lit(0))
).withColumn(
    "fraud_risk_score",
    col("amount_risk_score") +
    col("account_status_risk_score") +
    col("kyc_risk_score") +
    col("customer_risk_score") +
    col("model_flag_score")
).withColumn(
    "risk_category",
    when(col("fraud_risk_score") >= 70, lit("CRITICAL"))
    .when(col("fraud_risk_score") >= 40, lit("HIGH"))
    .when(col("fraud_risk_score") >= 20, lit("MEDIUM"))
    .otherwise(lit("LOW"))
)

fraud_risk_scores_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_FRAUD_RISK_TABLE)

print("Gold table created: fraud_risk_scores")

spark.table(GOLD_FRAUD_RISK_TABLE).display()

# ------------------------------------------------------------
# 7. Gold validation
# ------------------------------------------------------------

print("Gold table counts:")

spark.sql(f"""
SELECT 'daily_txn_summary' AS table_name, COUNT(*) AS row_count
FROM {GOLD_DAILY_SUMMARY_TABLE}

UNION ALL

SELECT 'customer_360' AS table_name, COUNT(*) AS row_count
FROM {GOLD_CUSTOMER_360_TABLE}

UNION ALL

SELECT 'fraud_risk_scores' AS table_name, COUNT(*) AS row_count
FROM {GOLD_FRAUD_RISK_TABLE}
""").display()


print("Risk category distribution:")

spark.sql(f"""
SELECT
    risk_category,
    COUNT(*) AS txn_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pct
FROM {GOLD_FRAUD_RISK_TABLE}
GROUP BY risk_category
ORDER BY txn_count DESC
""").display()


print("Top high-risk transactions:")

spark.sql(f"""
SELECT *
FROM {GOLD_FRAUD_RISK_TABLE}
ORDER BY fraud_risk_score DESC, amount DESC
LIMIT 50
""").display()